In [1]:
%pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import joblib
import time
import optuna

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

C:\Users\USER\Desktop\ai-fintech-platform\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
X_train = pd.read_pickle("../../data/processed/X_train.pkl")
X_test = pd.read_pickle("../../data/processed/X_test.pkl")

y_train = pd.read_pickle("../../data/processed/y_train.pkl")
y_test = pd.read_pickle("../../data/processed/y_test.pkl")

print("Data loaded successfully.")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

Data loaded successfully.
X_train: (472432, 421)
X_test: (118108, 421)


In [4]:
# ============================================================
# Optuna Objective Function for LightGBM
# ============================================================

def objective_lgbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1
    }

    model = LGBMClassifier(**params)

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]

    score = roc_auc_score(y_test, probabilities)

    return score

In [5]:
study_lgbm = optuna.create_study(
    direction="maximize",
    study_name="LightGBM_Optimization"
)

study_lgbm.optimize(
    objective_lgbm,
    n_trials=20,
    show_progress_bar=True
)

[I 2026-06-29 14:51:02,576] A new study created in memory with name: LightGBM_Optimization
  0%|          | 0/20 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.517084 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 37866
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 418
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

Best trial: 0. Best value: 0.967616:   5%|▌         | 1/20 [02:16<43:12, 136.43s/it]

[I 2026-06-29 14:53:19,198] Trial 0 finished with value: 0.9676161340198225 and parameters: {'n_estimators': 393, 'learning_rate': 0.042182281135350115, 'num_leaves': 269, 'max_depth': 13, 'min_child_samples': 30, 'subsample': 0.70141024933828, 'colsample_bytree': 0.8625081237273435, 'reg_alpha': 2.872096064128863, 'reg_lambda': 4.518584365840914}. Best is trial 0 with value: 0.9676161340198225.
[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.417568 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 37866
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 418
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


Best trial: 0. Best value: 0.967616:  10%|█         | 2/20 [03:10<26:27, 88.20s/it] 

...

[I 2026-06-29 15:36:05,926] Trial 17 finished with value: 0.9755491776051041 and parameters: {'n_estimators': 983, 'learning_rate': 0.17437201316294154, 'num_leaves': 184, 'max_depth': 12, 'min_child_samples': 39, 'subsample': 0.8605822447240018, 'colsample_bytree': 0.8454805107744083, 'reg_alpha': 0.7951074408942405, 'reg_lambda': 3.9463423846680317}. Best is trial 17 with value: 0.9755491776051041.
[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.361035 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 37866
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 418
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further spli

Best trial: 17. Best value: 0.975549:  95%|█████████▌| 19/20 [47:56<03:31, 211.09s/it]

[I 2026-06-29 15:38:58,923] Trial 18 finished with value: 0.9744740886047795 and parameters: {'n_estimators': 988, 'learning_rate': 0.17208096785529114, 'num_leaves': 113, 'max_depth': 13, 'min_child_samples': 59, 'subsample': 0.8532809283375199, 'colsample_bytree': 0.7936750660355805, 'reg_alpha': 0.7680620714674935, 'reg_lambda': 3.921644311316146}. Best is trial 17 with value: 0.9755491776051041.
[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.529962 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 37870
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 419
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

Best trial: 17. Best value: 0.975549: 100%|██████████| 20/20 [49:30<00:00, 148.52s/it]

[I 2026-06-29 15:40:33,184] Trial 19 finished with value: 0.9679097662302897 and parameters: {'n_estimators': 939, 'learning_rate': 0.1740909042037018, 'num_leaves': 187, 'max_depth': 6, 'min_child_samples': 23, 'subsample': 0.7648255309582799, 'colsample_bytree': 0.8222880711135648, 'reg_alpha': 0.7668556882567625, 'reg_lambda': 3.725210930840195}. Best is trial 17 with value: 0.9755491776051041.


In [6]:
print("Best ROC-AUC:", study_lgbm.best_value)
print("Best Parameters:")
study_lgbm.best_params

Best ROC-AUC: 0.9755491776051041
Best Parameters:


{'n_estimators': 983,
 'learning_rate': 0.17437201316294154,
 'num_leaves': 184,
 'max_depth': 12,
 'min_child_samples': 39,
 'subsample': 0.8605822447240018,
 'colsample_bytree': 0.8454805107744083,
 'reg_alpha': 0.7951074408942405,
 'reg_lambda': 3.9463423846680317}

In [7]:
best_lgbm_params = study_lgbm.best_params

best_lgbm_model = LGBMClassifier(
    **best_lgbm_params,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

best_lgbm_model.fit(X_train, y_train)

y_proba = best_lgbm_model.predict_proba(X_test)[:, 1]
y_pred = best_lgbm_model.predict(X_test)

print("Tuned LightGBM ROC-AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.733044 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 37866
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 418
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

In [9]:
print(confusion_matrix(y_test, y_pred))

[[113647    328]
 [   918   3215]]


In [10]:
joblib.dump(best_lgbm_model, "../../ml/models/lightgbm_tuned.pkl")
joblib.dump(study_lgbm, "../../ml/models/lightgbm_optuna_study.pkl")

print("Tuned LightGBM model and Optuna study saved.")

Tuned LightGBM model and Optuna study saved.
